# Error Handling and Cancellation

This notebook shows how exceptions propagate through SLURM and how to cancel running jobs.

You'll learn how to:
- Handle task exceptions that propagate from SLURM workers
- Handle partial failures in mapped tasks
- Cancel running tasks
- Work with `SlurmJobFailed` exceptions

## Setup

Make sure the Prefect server is running before executing this notebook:
```bash
pixi run prefect-start
```

In [ ]:
from prefect import flow, task
from prefect_submitit import SlurmJobFailed, SlurmTaskRunner

In [ ]:
# "local" or "slurm"
MODE = "slurm"

## Define Tasks

In [ ]:
@task
def fail_with(error_type: str, message: str):
    """Raises the specified exception type."""
    error_class = {"ValueError": ValueError, "RuntimeError": RuntimeError}[error_type]
    raise error_class(message)


@task
def conditional_fail(x: int, fail_on: int = -1) -> int:
    """Returns x * 10, but raises ValueError if x == fail_on."""
    if x == fail_on:
        raise ValueError(f"intentional failure on {x}")
    return x * 10


@task
def add(a: int, b: int) -> int:
    return a + b


@task
def sleep_and_return(seconds: float) -> float:
    import time
    time.sleep(seconds)
    return seconds

## Exception Propagation

When a task raises an exception on a SLURM worker, the original exception
is re-raised when you call `.result()` on the future.

In [ ]:
@flow(task_runner=SlurmTaskRunner(execution_mode=MODE, slurm_name="exception_flow"))
def exception_flow():
    future = fail_with.submit(error_type="ValueError", message="something went wrong")
    try:
        future.result()
    except ValueError as e:
        print(f"Caught ValueError: {e}")
        return str(e)


error_msg = exception_flow()
assert "something went wrong" in error_msg

## Partial Failures in Map

When mapping over inputs, some tasks may fail while others succeed.
Each future is independent — you can handle failures per-item.

In [ ]:
@flow(task_runner=SlurmTaskRunner(execution_mode=MODE, slurm_name="partial_failure_flow"))
def partial_failure_flow():
    # Task at x=3 will fail
    futures = conditional_fail.map(x=[1, 2, 3, 4, 5], fail_on=3)

    results = []
    for i, f in enumerate(futures):
        try:
            value = f.result()
            results.append(("ok", value))
            print(f"Item {i}: {value}")
        except ValueError as e:
            results.append(("error", str(e)))
            print(f"Item {i}: FAILED - {e}")

    return results


results = partial_failure_flow()

# Items 0,1,3,4 succeed; item 2 (x=3) fails
assert results[0] == ("ok", 10)
assert results[1] == ("ok", 20)
assert results[2][0] == "error"
assert results[3] == ("ok", 40)
assert results[4] == ("ok", 50)

## Cancel a Running Task

You can cancel a submitted task. This calls `scancel` on the SLURM job.
After cancellation, `.result()` raises `SlurmJobFailed`.

**Note:** In local mode, cancellation is a race with task completion.

In [ ]:
@flow(task_runner=SlurmTaskRunner(execution_mode=MODE, slurm_name="cancel_flow"))
def cancel_flow():
    # Submit a quick task and immediately cancel
    future = add.submit(1, 2)
    cancelled = future.cancel()

    try:
        result = future.result()
        # Task may have completed before cancel took effect
        print(f"Task completed before cancel: {result}")
        return ("completed", result)
    except SlurmJobFailed as e:
        print(f"Task was cancelled: {e}")
        return ("cancelled", None)


outcome, value = cancel_flow()
print(f"\nOutcome: {outcome}")

## Handling `SlurmJobFailed`

`SlurmJobFailed` is raised when a SLURM job fails at the infrastructure level
(not a Python exception in your code). This can happen due to:
- Job cancellation (`scancel`)
- Node failure
- Out of memory (OOM)
- Wall time exceeded

These are distinct from task-level Python exceptions, which are re-raised directly.

In [ ]:
@flow(task_runner=SlurmTaskRunner(execution_mode=MODE, slurm_name="robust_flow"))
def robust_flow():
    future = add.submit(1, 2)

    try:
        result = future.result()
        return result
    except SlurmJobFailed as e:
        # Infrastructure-level failure (cancelled, OOM, node fail, etc.)
        print(f"SLURM job failed: {e}")
        return None
    except Exception as e:
        # Task-level Python exception
        print(f"Task raised {type(e).__name__}: {e}")
        return None


result = robust_flow()
print(f"Result: {result}")